# MP0486 · RA1 · Tema 02 — Ejemplos de clase: persistencia y formatos

**Propósito.** Cada ejemplo resuelve un problema pequeño con datos ficticios. Acompaña los apuntes de rutas, ficheros, CSV, JSON y XML; el análisis tabular con Pandas se reserva para el Tema 03. Estos ejemplos resueltos **no son una práctica evaluable**.

Ejecuta las celdas **en orden**: se crean ficheros en `tema02_datos` bajo el directorio de trabajo de Colab, que es temporal. Ejecutar todo no pide teclado ni necesita instalar bibliotecas. Antes de ejecutar cada ejemplo, predice el resultado y después compruébalo.


## 1 · Preparar un espacio de trabajo para los datos
**Problema.** Una aplicación recibe datos brutos y genera ficheros procesados. Debe crear las carpetas necesarias sin escribir separadores de Windows o Linux en el código ni perderse si se ejecuta otra vez.

**Conceptos.** `Path.cwd()` indica el directorio de trabajo; `/` combina componentes de ruta; `mkdir(parents=True, exist_ok=True)` permite crear la estructura sin error al repetir la celda. Una ruta relativa depende del directorio de trabajo; `resolve()` muestra su ubicación absoluta.


In [ ]:
from pathlib import Path

base = Path.cwd() / 'tema02_datos'
brutos = base / 'brutos'
procesados = base / 'procesados'
brutos.mkdir(parents=True, exist_ok=True)
procesados.mkdir(parents=True, exist_ok=True)

print('Directorio de trabajo:', Path.cwd())
print('Datos brutos:', brutos.resolve())
print('Carpetas disponibles:', sorted(p.name for p in base.iterdir()))


**Comenta:** ¿por qué `'brutos/ventas.csv'` podría apuntar a otro lugar si cambia el directorio de trabajo? ¿Qué hace `exist_ok=True` cuando la carpeta ya existe?

## 2 · Recuperar texto escrito con otra codificación
**Problema.** Un fichero heredado contiene una palabra con tilde y se escribió con ISO-8859-1. Nuestra aplicación intenta leerlo como UTF-8; si la decodificación falla, prueba otra codificación candidata y muestra la lectura.

**Conceptos.** Un fichero almacena bytes; `encoding` indica cómo interpretarlos como texto. `with open(...)` cierra el recurso. `UnicodeDecodeError` significa que **esa interpretación** no funcionó. ISO-8859-1 puede decodificar cualquier byte: que no haya excepción **no prueba** que el texto sea correcto; hay que revisar su significado y procedencia.


In [ ]:
legado = brutos / 'nota_latin1.txt'
legado.write_text('Ciudad: Málaga', encoding='iso-8859-1')

def leer_con_codificaciones(ruta: Path, candidatas: tuple[str, ...]) -> tuple[str, str]:
    for codificacion in candidatas:
        try:
            with ruta.open('r', encoding=codificacion) as fichero:
                contenido = fichero.read()  # El error puede aparecer al leer, no al abrir.
            return contenido, codificacion
        except UnicodeDecodeError:
            print('No se pudo decodificar como', codificacion)
    raise UnicodeError('Ninguna codificación candidata permitió leer el fichero')

texto, candidata = leer_con_codificaciones(legado, ('utf-8', 'iso-8859-1'))
print('Codificación candidata:', candidata, '| Texto:', texto)
print('¿Aparece la ciudad esperada?', 'Málaga' in texto)


**Comenta:** ¿por qué la función lee **dentro** del `try`? Si los bytes se pudieran decodificar con las dos candidatas, ¿cómo decidirías cuál es la correcta?

## 3 · Guardar y ampliar un registro de incidencias
**Problema.** El programa debe escribir dos mensajes en un fichero de texto y añadir otro después sin borrar lo anterior. Finalmente los muestra uno por uno; no necesita cargar todo el fichero a la vez. También queremos releer la primera línea.

**Conceptos.** `w` crea o sobrescribe, `a` añade al final y `r` lee. `with` garantiza el cierre; iterar el fichero entrega líneas sucesivas. `readline()` lee una línea y `seek(0)` vuelve al inicio. Conviene indicar explícitamente `encoding='utf-8'`.


In [ ]:
registro = procesados / 'incidencias.txt'
with registro.open('w', encoding='utf-8') as salida:
    salida.write('Venta 1: aceptada\nVenta 2: pendiente\n')

with registro.open('a', encoding='utf-8') as salida:
    salida.write('Venta 3: aceptada\n')  # a no borra las dos líneas anteriores.

with registro.open('r', encoding='utf-8') as entrada:
    for numero, linea in enumerate(entrada, start=1):
        print(numero, linea.rstrip('\n'))  # El fichero incluye un salto de línea.

with registro.open('r', encoding='utf-8') as entrada:
    primera = entrada.readline()  # El puntero queda después de la línea leída.
    entrada.seek(0)  # Volvemos al inicio antes de releer.
    assert entrada.readline() == primera
print('La primera línea puede releerse tras seek(0).')


**Comenta:** predice qué pasaría si la segunda apertura usara `w` en lugar de `a`. ¿Por qué debemos tener cuidado con `w` en un fichero existente?

## 4 · Copiar bytes sin tratarlos como texto
**Problema.** Tenemos una pequeña secuencia de bytes, algunos no válidos como texto UTF-8. Debemos guardarla y recuperarla sin alterar su contenido. No pretendemos que sea una imagen real: solo ilustramos el tratamiento binario.

**Conceptos.** `wb` escribe bytes y `rb` los recupera. Interpretar datos binarios como texto requiere una codificación y puede producir un error o un resultado sin sentido. Para ficheros reales (PDF, PNG...) el contenido debe tratarse según su formato.


In [ ]:
muestra_bytes = bytes([0, 255, 10, 128])
ruta_binaria = brutos / 'muestra.bin'
with ruta_binaria.open('wb') as fichero:
    fichero.write(muestra_bytes)  # En modo binario write recibe bytes.

with ruta_binaria.open('rb') as fichero:
    recuperados = fichero.read()
print('Bytes originales:', muestra_bytes)
print('¿Se conservaron?', recuperados == muestra_bytes)
try:
    recuperados.decode('utf-8')
except UnicodeDecodeError:
    print('Esta secuencia no se interpreta como texto UTF-8.')


**Comenta:** ¿por qué cambiar la extensión `.bin` a `.txt` no convertiría el contenido en texto?

## 5 · Distinguir un fichero ausente de uno que ya existe
**Problema.** Antes de procesar una venta, queremos comunicar si falta su fichero. Cuando creamos un fichero de prueba por primera vez, no debemos sobrescribirlo por accidente al repetir el programa.

**Conceptos.** `FileNotFoundError` permite tratar una lectura imposible; `x` crea el fichero solo si no existía y, en otro caso, provoca `FileExistsError`. No se capturan errores ajenos a estos dos casos.


In [ ]:
pendiente = brutos / 'venta_inexistente.txt'
try:
    pendiente.read_text(encoding='utf-8')
except FileNotFoundError:
    print('No se puede procesar: falta', pendiente.name)

nuevo = brutos / 'creacion_exclusiva.txt'
try:
    with nuevo.open('x', encoding='utf-8') as fichero:
        fichero.write('Primera versión\n')
    print('Se creó un fichero nuevo.')
except FileExistsError:
    print('El fichero ya existía: no se sobrescribió.')


**Comenta:** ¿por qué `except Exception` sería menos informativo aquí que capturar `FileNotFoundError` o `FileExistsError`?

## 6 · Validar registros CSV y separar los rechazados
**Problema.** Una tabla de ventas separada por `;` contiene dos importes válidos, un `N/A` y un identificador incorrecto. Debemos convertir cada campo, guardar solo las filas válidas y conservar el motivo de los rechazos.

**Conceptos.** CSV contiene texto, no declara tipos numéricos. `csv.DictReader` usa nombres de columna como claves; `csv.DictWriter` escribe filas con una cabecera. `newline=''` al escribir evita problemas de líneas vacías en algunas plataformas. Se captura `ValueError` por fila sin perder las ventas restantes. En casos de dinero crítico conviene considerar `Decimal` y una política de precisión explícita; aquí `float` simplifica el primer contacto.


In [ ]:
import csv

origen_csv = brutos / 'ventas.csv'
origen_csv.write_text(
    'id;ciudad;importe\n'
    '1;Barcelona;12.50\n'
    '2;Girona;N/A\n'
    '3;Girona;20.00\n'
    'X;Barcelona;9.00\n', encoding='utf-8'
)
validas: list[dict] = []
rechazadas: list[dict] = []
with origen_csv.open('r', encoding='utf-8', newline='') as fichero:
    lector = csv.DictReader(fichero, delimiter=';')
    for numero_fila, fila in enumerate(lector, start=2):  # La fila 1 es la cabecera.
        try:
            venta = {'id': int(fila['id']), 'ciudad': fila['ciudad'].strip(),
                     'importe': float(fila['importe'].strip())}
            validas.append(venta)
        except (ValueError, TypeError, KeyError) as error:
            rechazadas.append({'fila': numero_fila, 'datos': fila,
                                'motivo': str(error)})

salida_csv = procesados / 'ventas_validas.csv'
with salida_csv.open('w', encoding='utf-8', newline='') as fichero:
    escritor = csv.DictWriter(fichero, fieldnames=['id', 'ciudad', 'importe'], delimiter=';')
    escritor.writeheader()
    escritor.writerows(validas)
print('Aceptadas:', len(validas), '| Rechazadas:', len(rechazadas))
for rechazo in rechazadas:
    print('Fila rechazada:', rechazo['fila'], '| Motivo:', rechazo['motivo'])
print('CSV resultante:')
print(salida_csv.read_text(encoding='utf-8'))


**Comenta:** ¿por qué la fila con `N/A` no debe convertirse automáticamente en una venta de importe cero? ¿Qué cabecera tiene el CSV de salida?

## 7 · Intercambiar ventas en JSON sin perder los tipos
**Problema.** Otra aplicación solicita las ventas válidas en JSON. Debemos guardar una lista de objetos con números y texto, releerla y reconocer un documento JSON mal formado sin interrumpir la lectura de otros ficheros.

**Conceptos.** `json.dump` escribe un objeto Python; `json.load` lee un fichero. En JSON, los números y cadenas mantienen tipos distintos; `None` se representa como `null`. `ensure_ascii=False` hace legibles las tildes y `indent=2` facilita inspeccionar el fichero. Un documento corrupto provoca `JSONDecodeError`.


In [ ]:
import json

ruta_json = procesados / 'ventas.json'
with ruta_json.open('w', encoding='utf-8') as fichero:
    json.dump(validas, fichero, ensure_ascii=False, indent=2)
with ruta_json.open('r', encoding='utf-8') as fichero:
    ventas_json = json.load(fichero)
print('Primera venta:', ventas_json[0])
print('Tipo de id:', type(ventas_json[0]['id']).__name__)
print('¿Contenido conservado?', ventas_json == validas)

try:
    json.loads('{"ventas": [}')  # El array no contiene un valor válido.
except json.JSONDecodeError as error:
    print('JSON inválido, posición:', error.pos)


**Comenta:** ¿qué pasaría si el origen fuese directamente el CSV bruto y no convirtiéramos `id` e `importe` antes de exportar a JSON?

## 8 · Representar las ventas como un árbol XML
**Problema.** Un sistema heredado exige un documento `<ventas>` con un nodo `<venta>` por registro y tres elementos hijos: `id`, `ciudad` e `importe`. Debemos escribirlo, recuperar las ventas y detectar XML mal formado.

**Conceptos.** XML tiene raíz, elementos y una estructura jerárquica; `ElementTree` construye y analiza el árbol. `ET.parse` lee XML y `ET.ParseError` identifica problemas sintácticos. El texto leído de XML **no recupera tipos Python automáticamente**: aquí volvemos a convertir `id` e `importe`. XML de orígenes externos no confiables requiere límites y medidas adicionales.


In [ ]:
import xml.etree.ElementTree as ET

raiz = ET.Element('ventas')
for venta in validas:
    nodo = ET.SubElement(raiz, 'venta')  # Una fila se representa como nodo hijo.
    ET.SubElement(nodo, 'id').text = str(venta['id'])
    ET.SubElement(nodo, 'ciudad').text = venta['ciudad']
    ET.SubElement(nodo, 'importe').text = str(venta['importe'])

ruta_xml = procesados / 'ventas.xml'
ET.ElementTree(raiz).write(ruta_xml, encoding='utf-8', xml_declaration=True)
recuperadas_xml: list[dict] = []
for nodo in ET.parse(ruta_xml).getroot().findall('venta'):
    recuperadas_xml.append({'id': int(nodo.findtext('id')),
                            'ciudad': nodo.findtext('ciudad'),
                            'importe': float(nodo.findtext('importe'))})
print('XML recuperado:', recuperadas_xml)
print('¿Información conservada?', recuperadas_xml == validas)

try:
    ET.fromstring('<ventas><venta></ventas>')
except ET.ParseError:
    print('XML mal formado: las etiquetas no coinciden.')


**Comenta:** ¿por qué el fichero XML guarda `12.5` como texto aunque el diccionario Python contenía un `float`? ¿Qué pasaría si faltara el nodo `<id>`?

## 9 · Comprobar el viaje de ida y vuelta de los datos
**Problema.** Tras generar CSV, JSON y XML, debemos verificar que todos conservan las dos ventas aceptadas, su ciudad y sus importes. Contar filas no basta: también necesitamos comparar campos y tipos.

**Conceptos.** Al leer CSV y XML se restauran tipos según una regla explícita; JSON ya conserva tipos básicos. `assert` permite expresar un contrato pequeño; en un proyecto real, las comprobaciones se integrarían en tests y se documentarían las conversiones que puedan perder información.


In [ ]:
recuperadas_csv: list[dict] = []
with salida_csv.open('r', encoding='utf-8', newline='') as fichero:
    for fila in csv.DictReader(fichero, delimiter=';'):
        recuperadas_csv.append({'id': int(fila['id']), 'ciudad': fila['ciudad'],
                                'importe': float(fila['importe'])})

assert len(validas) == 2  # Deben conservarse exactamente las aceptadas.
assert recuperadas_csv == ventas_json == recuperadas_xml == validas
assert len(rechazadas) == 2  # Las otras dos siguen registradas aparte.
print('CSV, JSON y XML conservan las ventas válidas y sus tipos.')
print('Rechazos documentados:', [r['fila'] for r in rechazadas])


**Comenta:** las comprobaciones pasan en este caso concreto. ¿Qué otra prueba añadirías para detectar un campo perdido? ¿Por qué una conversión general entre formatos jerárquicos y tabulares no siempre puede conservar toda la información?

## Cierre
Hemos resuelto problemas de rutas, texto y bytes, codificaciones, modos de apertura, errores, validación CSV y conversiones JSON/XML. Todas las ventas y datos son ficticios y los ficheros se generan localmente. Los ejemplos se estudian y comentan; **las prácticas posteriores** pedirán crear procesos nuevos con datos y requisitos distintos, pudiendo usar IA siempre que comprendas, compruebes y justifiques sus propuestas.

**Siguiente paso:** Tema 03 — inspeccionar, limpiar, agrupar y visualizar datos tabulares con Pandas.
